In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
import warnings
import os
warnings.filterwarnings('ignore')

# Load data
data = pd.read_excel('data.xlsx')
res = data.values

# Data analysis
num_size = 0.8  # training set proportion
outdim = 1  # last column is the output
num_samples = res.shape[0]
f_ = res.shape[1] - outdim  # input feature dimension

# Split training and test sets
num_train_s = round(num_size * num_samples)
P_train = res[:num_train_s, :f_].T
T_train = res[:num_train_s, f_:].T
M = P_train.shape[1]

P_test = res[num_train_s:, :f_].T
T_test = res[num_train_s:, f_:].T
N = P_test.shape[1]

# Data cleaning - remove data points containing NaN
train_nan_mask = np.any(np.isnan(P_train), axis=0) | np.any(np.isnan(T_train), axis=0)
P_train = P_train[:, ~train_nan_mask]
T_train = T_train[:, ~train_nan_mask]
M = P_train.shape[1]
print(f'Number of training samples after cleaning: {M}')

test_nan_mask = np.any(np.isnan(P_test), axis=0) | np.any(np.isnan(T_test), axis=0)
P_test = P_test[:, ~test_nan_mask]
T_test = T_test[:, ~test_nan_mask]
N = P_test.shape[1]

# Data normalization
input_scaler = MinMaxScaler()
output_scaler = MinMaxScaler()

inputn_train = input_scaler.fit_transform(P_train.T).T
inputn_test = input_scaler.transform(P_test.T).T

outputn_train = output_scaler.fit_transform(T_train.T).T
outputn_test = output_scaler.transform(T_test.T).T

# Save normalization parameters
np.save('input_scaler_min.npy', input_scaler.min_)
np.save('input_scaler_scale.npy', input_scaler.scale_)
np.save('output_scaler_min.npy', output_scaler.min_)
np.save('output_scaler_scale.npy', output_scaler.scale_)

# Prepare training and test data
XrTrain = [inputn_train[:, i] for i in range(inputn_train.shape[1])]
YrTrain = [outputn_train[:, i] for i in range(outputn_train.shape[1])]

XrTest = [inputn_test[:, i] for i in range(inputn_test.shape[1])]
YrTest = [outputn_test[:, i] for i in range(outputn_test.shape[1])]

# Convert to TensorFlow datasets
train_dataset = tf.data.Dataset.from_tensor_slices((XrTrain, YrTrain)).batch(32)
test_dataset = tf.data.Dataset.from_tensor_slices((XrTest, YrTest)).batch(32)

# Create model
numFeatures = P_train.shape[0]
numResponses = 1
FiltZise = 4

# Build model
inputs = tf.keras.Input(shape=(numFeatures,))
x = layers.Reshape((numFeatures, 1))(inputs)

# CNN feature extraction
x = layers.Conv1D(128, FiltZise, padding='same', kernel_initializer='he_normal')(x)
x = layers.BatchNormalization()(x)
x = layers.ReLU()(x)

# Pooling layer
x = layers.AveragePooling1D(pool_size=FiltZise, strides=FiltZise)(x)

# Flatten layer
x = layers.Flatten()(x)

# Dropout
x = layers.Dropout(0.1)(x)

# Fully connected layer
outputs = layers.Dense(numResponses)(x)

model = tf.keras.Model(inputs=inputs, outputs=outputs)

# Compile model - use explicit loss function objects instead of strings
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=tf.keras.losses.MeanSquaredError(),
    metrics=[tf.keras.metrics.MeanAbsoluteError()]
)

# Learning rate scheduler
lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(
    factor=0.1,
    patience=50,
    verbose=1
)

# Model checkpoint callback - save best model
checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    'best_model.h5',
    monitor='val_loss',
    save_best_only=True,
    mode='min',
    verbose=1
)

# Train model
history = model.fit(
    train_dataset,
    epochs=500,
    validation_data=test_dataset,
    callbacks=[lr_scheduler, checkpoint_callback],
    verbose=1
)

# Save final model - use compatible saving method
# Method 1: Save weights and architecture separately
#model_json = model.to_json()
#with open("model_architecture.json", "w") as json_file:
    #json_file.write(model_json)
#model.save_weights("model_weights.h5")
#print("Model architecture and weights saved as model_architecture.json and model_weights.h5")

# Method 2: Use compatible save format
model.save('model.h5', save_format='h5')
print("Model saved as model.h5")

# Save training history
history_df = pd.DataFrame(history.history)
history_df.to_csv('training_history.csv', index=False)
print("Training history saved as training_history.csv")

# Create model architecture visualization
tf.keras.utils.plot_model(
    model,
    to_file='model_architecture.png',
    show_shapes=True,
    show_layer_names=True
)
print("Model architecture diagram saved as model_architecture.png")

# Evaluate model
test_loss, test_mae = model.evaluate(test_dataset, verbose=0)
print(f"Test loss: {test_loss:.4f}, Test MAE: {test_mae:.4f}")

print("Training complete, model and related files saved")